## Snorkel Programmatic Data Annotation (IMDB subset ~5,000)
\n
This notebook installs dependencies, loads the Hugging Face IMDB dataset (small subset), defines **5 conflicting labeling functions**, analyzes their overlaps/conflicts, then compares **MajorityLabelVoter** vs **LabelModel** aggregation.

In [1]:
# 1) Setup (Colab installs)\n
!pip -q install snorkel datasets pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 3.6 MB/s eta 0:00:00


In [11]:
import re
import numpy as np
import pandas as pd

from datasets import load_dataset

from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import MajorityLabelVoter, LabelModel

ABSTAIN = -1
NEGATIVE = 0
POSITIVE = 1

In [12]:
ds = load_dataset("imdb")

# Keep it fast: shuffle + small subset
df = ds["train"].shuffle(seed=42).select(range(5000)).to_pandas()
df = df.rename(columns={"text": "review", "label": "gold_label"})

# IMDB labels are already 0=neg, 1=pos; we map explicitly for clarity
df["gold_label"] = df["gold_label"].map({0: NEGATIVE, 1: POSITIVE})

print(df.head(3))
print("\nLabel distribution (gold):")
print(df["gold_label"].value_counts())

                                              review  gold_label
0  There is no relation at all between Fortier an...           1
1  This movie is a great. The plot is very true t...           1
2  George P. Cosmatos' "Rambo: First Blood Part I...           0

Label distribution (gold):
gold_label
1    2506
0    2494
Name: count, dtype: int64


In [13]:
def _has_any_word(text, vocab):
    t = text.lower()
    return any(re.search(rf"\b{re.escape(w)}\b", t) for w in vocab)

def _has_negation_before(text, word, window=3):
    t = text.lower()
    pattern = rf"\b(not|never|no)\b(?:\W+\w+){{0,{window}}}\W+\b{re.escape(word)}\b"
    return re.search(pattern, t) is not None

@labeling_function()
def lf_pos_good_great(x):
    if _has_any_word(x.review, {"good", "great", "excellent", "amazing"}):
        return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_neg_bad_terrible(x):
    if _has_any_word(x.review, {"bad", "terrible", "awful", "boring"}):
        return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_negated_positive_is_negative(x):
    for w in ["good", "great", "excellent", "amazing"]:
        if _has_negation_before(x.review, w):
            return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_negated_negative_is_positive(x):
    for w in ["bad", "terrible", "awful", "boring"]:
        if _has_negation_before(x.review, w):
            return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_exclamation_heuristic(x):
    if x.review.count("!") >= 3:
        return POSITIVE
    return ABSTAIN

lfs = [
    lf_pos_good_great,
    lf_neg_bad_terrible,
    lf_negated_positive_is_negative,
    lf_negated_negative_is_positive,
    lf_exclamation_heuristic,
]

print("Defined LFs:", [lf.name for lf in lfs])

Defined LFs: ['lf_pos_good_great', 'lf_neg_bad_terrible', 'lf_negated_positive_is_negative', 'lf_negated_negative_is_positive', 'lf_exclamation_heuristic']


In [14]:
applier = PandasLFApplier(lfs=lfs)
L = applier.apply(df=df)

analysis = LFAnalysis(L=L, lfs=lfs).lf_summary()
print("=== LFAnalysis summary ===")
print(analysis)

print("\n=== Conflicts column (explicit proof of disagreement) ===")
# Corrected column name from 'Conflict' to 'Conflicts'
print(analysis[['Conflicts']])

candidate_rows = np.where((L != ABSTAIN).sum(axis=1) > 0)[0]
point_conflicts = []
for i in candidate_rows:
    labels = L[i]
    non_abstain = labels[labels != ABSTAIN]
    if len(non_abstain) >= 2 and len(set(non_abstain.tolist())) > 1:
        point_conflicts.append(i)

print(f"\nNumber of data points with LF conflicts: {len(point_conflicts)}")
if len(point_conflicts) > 0:
    ex_i = point_conflicts[0]
    print("\nExample conflicting review (first found):")
    print(df.loc[ex_i, "review"][:600], "...")
    print("LF outputs:", L[ex_i].tolist())

100%|██████████| 5000/5000 [00:02<00:00, 1711.16it/s]

=== LFAnalysis summary ===
                                 j Polarity  Coverage  Overlaps  Conflicts
lf_pos_good_great                0      [1]    0.5692    0.2514     0.2126
lf_neg_bad_terrible              1      [0]    0.3174    0.2112     0.2112
lf_negated_positive_is_negative  2      [0]    0.0494    0.0494     0.0494
lf_negated_negative_is_positive  3      [1]    0.0302    0.0302     0.0302
lf_exclamation_heuristic         4      [1]    0.1136    0.0868     0.0480

=== Conflicts column (explicit proof of disagreement) ===
                                 Conflicts
lf_pos_good_great                   0.2126
lf_neg_bad_terrible                 0.2112
lf_negated_positive_is_negative     0.0494
lf_negated_negative_is_positive     0.0302
lf_exclamation_heuristic            0.0480

Number of data points with LF conflicts: 1184

Example conflicting review (first found):
I give this movie 7 out of 10 because the villains were interesting in their roles and the unknown batwoman creates 

In [15]:
majority_model = MajorityLabelVoter(cardinality=2)
maj_labels = majority_model.predict(L=L)
print("MajorityLabelVoter labels (first 10):", maj_labels[:10].tolist())

MajorityLabelVoter labels (first 10): [1, 1, 0, -1, 1, 1, -1, 0, -1, 1]


In [16]:
label_model = LabelModel(cardinality=2, verbose=False)
label_model.fit(L_train=L, n_epochs=200, log_freq=50, seed=42)
lm_labels = label_model.predict(L=L)
print("LabelModel labels (first 10):", lm_labels[:10].tolist())

100%|██████████| 200/200 [00:00<00:00, 1066.52epoch/s]


LabelModel labels (first 10): [1, 1, 0, -1, 1, 1, 1, 0, 1, 1]


In [17]:
comparison = pd.DataFrame({
    "review_snippet": df["review"].str.replace(r"\s+", " ", regex=True).str.slice(0, 120),
    "gold_label": df["gold_label"],
    "majority_vote": maj_labels,
    "label_model": lm_labels,
})
comparison["maj_vs_lm_diff"] = (comparison["majority_vote"] != comparison["label_model"]).astype(int)

display(comparison.head(20))
print(f"\nDisagreement rate (Majority vs LabelModel): {comparison['maj_vs_lm_diff'].mean():.2%}")

,review_snippet,gold_label,majority_vote,label_model,maj_vs_lm_diff
0,There is no relation at all between Fortier an...,1,1,1,0
1,This movie is a great. The plot is very true t...,1,1,1,0
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0,0,0,0
3,In the process of trying to establish the audi...,1,-1,-1,0
4,"Yeh, I know -- you're quivering with excitemen...",0,1,1,0
5,While this movie's style isn't as understated ...,1,1,1,0
6,I give this movie 7 out of 10 because the vill...,1,-1,1,1
7,"really awful... lead actor did OK... the film,...",0,0,0,0
8,Good grief I can't even begin to describe how ...,0,-1,1,1
9,Home Room deals with a Columbine-like high-sch...,1,1,1,0



Disagreement rate (Majority vs LabelModel): 20.76%
